In [1]:
# ================================
# exp29_elasticnet_derivative_stacking
# ElasticNet + spectral derivative + stacking ensemble
# ================================

import pandas as pd
import numpy as np
import os

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error


# ================================
# Metric diagnostics tool
# ================================

def metric_diagnostics(y_true, y_pred):

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    y_true_log = np.log1p(y_true)
    y_pred_log = np.log1p(np.maximum(y_pred,0))
    rmsle = np.sqrt(mean_squared_error(y_true_log, y_pred_log))

    nrmse_mean = rmse / np.mean(y_true)
    nrmse_range = rmse / (np.max(y_true) - np.min(y_true))

    print("\nMetric diagnostics")
    print("------------------")
    print("RMSE:", rmse)
    print("MAE:", mae)
    print("RMSLE:", rmsle)
    print("NRMSE (mean):", nrmse_mean)
    print("NRMSE (range):", nrmse_range)


# ================================
# Spectral derivative function
# ================================

def spectral_derivative(df):

    derivative = df.diff(axis=1)
    derivative = derivative.fillna(0)

    derivative.columns = [f"{c}_d1" for c in df.columns]

    return derivative


# ================================
# Load data
# ================================

train = pd.read_csv("../data/train.csv", encoding="cp932")
test = pd.read_csv("../data/test.csv", encoding="cp932")

target = "含水率"
id_col = "sample number"

spectral_cols = [
    c for c in train.columns
    if c not in ["sample number","species number","樹種","含水率"]
]

X_raw = train[spectral_cols]
X_test_raw = test[spectral_cols]

y = train[target]


# ================================
# Create derivative features
# ================================

X_deriv = spectral_derivative(X_raw)
X_test_deriv = spectral_derivative(X_test_raw)

X = pd.concat([X_raw, X_deriv], axis=1)
X_test = pd.concat([X_test_raw, X_test_deriv], axis=1)

print("Original features:", X_raw.shape[1])
print("Derivative features:", X_deriv.shape[1])
print("Total features:", X.shape[1])


# ================================
# ElasticNet configurations
# ================================

elastic_configs = [
    (0.0005, 0.9),
    (0.001, 0.9),
    (0.002, 0.9),
    (0.001, 0.8)
]


# ================================
# KFold
# ================================

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_predictions = np.zeros(len(X))
test_predictions = np.zeros(len(X_test))


# ================================
# Training loop
# ================================

for alpha, l1_ratio in elastic_configs:

    print(f"\nTraining ElasticNet alpha={alpha}, l1_ratio={l1_ratio}")

    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet(
            alpha=alpha,
            l1_ratio=l1_ratio,
            max_iter=50000,
            random_state=42
        ))
    ])

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model.fit(X_train, y_train)

        pred_val = model.predict(X_val)

        oof[val_idx] = pred_val

        test_pred += model.predict(X_test) / kf.n_splits

    metric_diagnostics(y, oof)

    oof_predictions += oof / len(elastic_configs)
    test_predictions += test_pred / len(elastic_configs)


# ================================
# Final diagnostics
# ================================

print("\nFinal Ensemble Performance")
metric_diagnostics(y, oof_predictions)


# ================================
# Save submission
# ================================

os.makedirs("../submissions", exist_ok=True)

submission = pd.DataFrame({
    id_col: test[id_col],
    target: test_predictions
})

output_path = "../submissions/exp29_elasticnet_derivative_stacking.csv"

submission.to_csv(output_path, index=False, header=False)

print("\nSubmission saved:", output_path)

Original features: 1555
Derivative features: 1555
Total features: 3110

Training ElasticNet alpha=0.0005, l1_ratio=0.9


/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.644e+03, tolerance: 2.646e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.596e+03, tolerance: 2.477e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check t


Metric diagnostics
------------------
RMSE: 13.742899370279906
MAE: 10.144826142239031
RMSLE: 0.7256595139526709
NRMSE (mean): 0.2752010418248412
NRMSE (range): 0.046157193746935804

Training ElasticNet alpha=0.001, l1_ratio=0.9


/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.050e+03, tolerance: 2.646e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.359e+03, tolerance: 2.477e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check t


Metric diagnostics
------------------
RMSE: 12.528998880642153
MAE: 9.376196079731706
RMSLE: 0.6699224777023891
NRMSE (mean): 0.25089273028015796
NRMSE (range): 0.042080161777184225

Training ElasticNet alpha=0.002, l1_ratio=0.9


/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.118e+03, tolerance: 2.646e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.848e+03, tolerance: 2.477e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check t


Metric diagnostics
------------------
RMSE: 11.101664128143344
MAE: 8.472682456644536
RMSLE: 0.6111564608544856
NRMSE (mean): 0.22231040566749702
NRMSE (range): 0.037286284958490734

Training ElasticNet alpha=0.001, l1_ratio=0.8


/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.361e+03, tolerance: 2.646e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.761e+03, tolerance: 2.477e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check t


Metric diagnostics
------------------
RMSE: 12.434970271497532
MAE: 9.206028728105352
RMSLE: 0.6490241186419721
NRMSE (mean): 0.24900981092662614
NRMSE (range): 0.041764355293187914

Final Ensemble Performance

Metric diagnostics
------------------
RMSE: 12.009881459545603
MAE: 9.009024173765424
RMSLE: 0.6534365763672384
NRMSE (mean): 0.24049742349182882
NRMSE (range): 0.040336642979776696

Submission saved: ../submissions/exp29_elasticnet_derivative_stacking.csv


/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.648e+03, tolerance: 2.606e+02
  model = cd_fast.enet_coordinate_descent(
